# Route Resilience — Segmentation Fine-Tuning (task A4)

Fine-tune a **SegFormer MiT-B0 encoder + U-Net decoder** for occlusion-robust
road extraction, on the **DeepGlobe Roads** corpus. Runs on **Google Colab** or
**Kaggle** (auto-detected); both give a free 16 GB GPU.

**Pipeline role:** P1 — produces the binary road masks that P2 skeletonises.
The reusable logic lives in `src/pipeline/p1_segment/` (model, dataset, losses,
metrics, train) and is unit-tested on CPU; this notebook is the orchestrator.

**Rules honoured:** fine-tune pretrained only (ImageNet encoder), PyTorch only,
AMP for 8 GB-class memory, checkpoint saved **off-device** (cloud is wiped).

**Done-when (Tracker A4):** model outputs a real road mask; **IoU** and
**Occlusion-Recall** logged.

In [ ]:
# 1 · Detect environment (Kaggle / Colab / local)
import os, sys

def detect_env():
    # Check Kaggle FIRST: some Kaggle images also have /content, which would
    # otherwise trip the Colab check below and pick the wrong dataset paths.
    if os.path.exists("/kaggle") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if "google.colab" in sys.modules or os.path.exists("/content"):
        return "colab"
    return "local"

ENV = detect_env()
print("environment:", ENV)

In [ ]:
# 2 · Install dependencies (the GPU wheel index differs by platform)
# Colab/Kaggle ship torch; we add the segmentation + augmentation libs.
#
# NOTE (Kaggle): enable networking first — Settings (right sidebar) -> Internet -> On
# (needs a phone-verified account). Without it, this pip install, the git clone in
# cell 3, and the pretrained-weights download in cell 6 all fail with a DNS /
# "name resolution" error. Colab has internet on by default.
%pip install -q segmentation-models-pytorch==0.3.4 albumentations==2.0.8 albucore==0.0.24 opencv-python-headless==4.10.0.84
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0))

In [ ]:
# 3 · Get the project source so we can import src.pipeline.p1_segment.*
# All active work lives on the `dev` integration branch; `main` is the stage-gate
# and stays near-empty until a release, so we must clone `dev` (not the default).
REPO_URL = "https://github.com/Akshat-Tiwari69/Trace.git"
REPO_BRANCH = "dev"
REPO_DIR = "/content/Trace" if ENV == "colab" else ("/kaggle/working/Trace" if ENV == "kaggle" else ".")

if ENV in ("colab", "kaggle") and not os.path.exists(os.path.join(REPO_DIR, "src", "pipeline")):
    get_ipython().system(f"rm -rf {REPO_DIR}")  # clear any partial/wrong-branch clone
    get_ipython().system(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.pipeline.p1_segment.model import build_model, save_checkpoint, predict_mask
from src.pipeline.p1_segment.dataset import (
    RoadTileDataset, pair_deepglobe, build_train_transform, build_val_transform)
from src.pipeline.p1_segment.losses import DiceBCELoss
from src.pipeline.p1_segment.metrics import iou_score, dice_score, occlusion_recall
from src.pipeline.p1_segment.train import train_one_epoch, evaluate
print("imported p1_segment modules OK")

## 4 · Dataset — DeepGlobe Roads

DeepGlobe pairs are `{id}_sat.jpg` (image) + `{id}_mask.png` (label).

- **Kaggle:** add the *DeepGlobe Road Extraction Dataset* via *+ Add Data*; it
  mounts under `/kaggle/input/...`. Set `DATA_ROOT` to the `train/` folder.
- **Colab:** upload your `kaggle.json` and pull the same dataset, or mount Drive
  and point `DATA_ROOT` at a `train/` folder of `*_sat.jpg` / `*_mask.png`.

In [ ]:
# 4 · Configure data paths + hyper-parameters (config, not hardcoded)
import glob
from pathlib import Path

CONFIG = {
    "tile_size": 256,
    "encoder": "mit_b2",       # bigger SegFormer backbone (~24M) — top accuracy lever; fits a T4
    "batch_size": 16,          # split across GPUs if >1 (8/GPU on 2x T4)
    "crops_per_image": 4,      # use ~4 random 256 windows of each 1024 tile per epoch (data efficiency)
    "epochs": 12,              # mit_b2 + 4x data is slower — best ckpt saves each epoch, so safe to stop early
    "lr": 3e-4,                # peak LR; cosine-decayed in cell 7
    "val_fraction": 0.1,
    "threshold": 0.5,          # tuned on val in cell 7b
    "occlusion": True,         # CoarseDropout occlusion-sim augmentation
    "cldice_weight": 0.3,      # soft-clDice connectivity loss weight (0 = plain Dice+BCE)
    "seed": 42,
}


def _has_pairs(d: str) -> bool:
    # DeepGlobe valid/ and test/ have *_sat.jpg but NO masks — require both.
    return bool(glob.glob(os.path.join(d, "*_sat.jpg"))
                and glob.glob(os.path.join(d, "*_mask.png")))


def find_train_dir() -> str:
    """Find the DeepGlobe dir with BOTH images and masks, preferring 'train'.
    Doesn't hardcode the Kaggle mount path (it varies by dataset slug)."""
    if ENV == "kaggle":
        candidates = (glob.glob("/kaggle/input/**/train", recursive=True)   # prefer train/
                      + glob.glob("/kaggle/input/**/", recursive=True))
    elif ENV == "colab":
        candidates = (["/content/deepglobe/train", "/content/train"]
                      + glob.glob("/content/**/train", recursive=True))
    else:
        candidates = ["data/raw/deepglobe/train"]
    for c in candidates:
        if _has_pairs(c.rstrip("/")):
            return c.rstrip("/")
    return candidates[0].rstrip("/") if candidates else ""


DATA_ROOT = find_train_dir()
CKPT_DIR = Path(REPO_DIR) / "models"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
n_sat = len(glob.glob(os.path.join(DATA_ROOT, "*_sat.jpg")))
n_mask = len(glob.glob(os.path.join(DATA_ROOT, "*_mask.png")))
print(f"DATA_ROOT: {DATA_ROOT}\n  _sat.jpg: {n_sat} | _mask.png: {n_mask} | pairs ready: {_has_pairs(DATA_ROOT)}")
if not _has_pairs(DATA_ROOT):
    print("  no image+mask pairs — add the DeepGlobe *dataset* (balraj98) via + Add Data.",
          "Inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "n/a")

In [ ]:
# 5 · Build (image, mask) pairs, split train/val, make loaders
import random
from torch.utils.data import DataLoader

random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])

pairs = pair_deepglobe(DATA_ROOT)
assert pairs, f"no *_sat.jpg/_mask.png pairs under {DATA_ROOT} - check DATA_ROOT"
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * CONFIG["val_fraction"]))
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print(f"{len(train_pairs)} train x {CONFIG['crops_per_image']} crops / {len(val_pairs)} val pairs")

# train: multiple random native-res crops per image/epoch; val: one centre crop
train_ds = RoadTileDataset(
    train_pairs, build_train_transform(CONFIG["tile_size"], CONFIG["occlusion"]),
    crops_per_image=CONFIG["crops_per_image"])
val_ds = RoadTileDataset(val_pairs, build_val_transform(CONFIG["tile_size"]))

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"],
                        num_workers=4, pin_memory=True)

In [ ]:
# 6 · Model + loss + optimizer + AMP scaler (+ multi-GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(encoder=CONFIG["encoder"], encoder_weights="imagenet").to(device)

n_gpu = torch.cuda.device_count()
if n_gpu > 1:                       # use both T4s — DataParallel splits each batch across GPUs
    model = torch.nn.DataParallel(model)
    print(f"DataParallel across {n_gpu} GPUs")

loss_fn = DiceBCELoss(bce_weight=0.5, cldice_weight=CONFIG["cldice_weight"])  # +soft-clDice
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
print(f"model: {CONFIG['encoder']} on {device} | params (M):",
      round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
      f"| clDice w={CONFIG['cldice_weight']}")

In [ ]:
# 7 · Fine-tune - cosine-decayed LR, log IoU + Dice, keep the best checkpoint
from torch.optim.lr_scheduler import CosineAnnealingLR

# Start cosine from the configured peak LR every run, so re-running this cell
# doesn't inherit an already-decayed LR. (For a fully clean restart re-run cell 6.)
for g in optimizer.param_groups:
    g["lr"] = CONFIG["lr"]
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
CKPT_PATH = CKPT_DIR / f"segformer_{CONFIG['encoder']}_deepglobe.pt"
best_iou = -1.0
for epoch in range(1, CONFIG["epochs"] + 1):
    cur_lr = optimizer.param_groups[0]["lr"]   # LR actually used THIS epoch
    tr_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device, scaler)
    scores = evaluate(model, val_loader, device, CONFIG["threshold"])
    print(f"epoch {epoch:02d} | loss {tr_loss:.4f} | IoU {scores['iou']:.4f} | "
          f"Dice {scores['dice']:.4f} | lr {cur_lr:.2e}")
    if scores["iou"] > best_iou:
        best_iou = scores["iou"]
        save_checkpoint(model, CKPT_PATH,
                        meta={"encoder": CONFIG["encoder"], "epoch": epoch,
                              "iou": scores["iou"], "dice": scores["dice"],
                              "tile_size": CONFIG["tile_size"]})
    scheduler.step()   # advance LR for the NEXT epoch (after logging this one)
print("best val IoU:", round(best_iou, 4), "->", CKPT_PATH)

In [ ]:
# 7b · Tune the decision threshold on val (roads are rare -> best cutoff is often < 0.5)
import numpy as np
ths = np.round(np.arange(0.20, 0.61, 0.05), 2)
inter = {float(t): 0.0 for t in ths}
union = {float(t): 0.0 for t in ths}
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        probs = torch.sigmoid(model(images.to(device))).cpu()
        masks = masks.float()
        for t in ths:
            t = float(t)
            pred = (probs >= t).float()
            i = (pred * masks).sum().item()
            inter[t] += i
            union[t] += pred.sum().item() + masks.sum().item() - i
ious = {t: inter[t] / max(union[t], 1e-7) for t in inter}
best_t = max(ious, key=ious.get)
print("IoU by threshold:", {t: round(v, 4) for t, v in ious.items()})
print(f"best threshold {best_t} -> IoU {ious[best_t]:.4f}  (0.50 -> {ious[0.5]:.4f})")
CONFIG["threshold"] = float(best_t)   # used by cells 8 and 9

## 8 · Occlusion-Recall — does the model see through occlusion?

We re-apply CoarseDropout to validation tiles, recording the dropout boxes as an
*occlusion mask*, then measure recall **only inside** those boxes
(`occlusion_recall`). High = the model recovers roads it cannot directly see.

In [ ]:
# 8 · Occlusion-Recall on the validation set
import numpy as np, cv2, albumentations as A
from src.pipeline.p1_segment.model import IMAGENET_MEAN, IMAGENET_STD
from albumentations.pytorch import ToTensorV2

S = CONFIG["tile_size"]
# native-resolution centre crop (same scale as training), then occlude
occ_aug = A.Compose([
    A.PadIfNeeded(S, S, border_mode=0),
    A.CenterCrop(S, S),
    A.CoarseDropout(num_holes_range=(1, 8),
                    hole_height_range=(S // 16, S // 8),
                    hole_width_range=(S // 16, S // 8),
                    fill=0, fill_mask=None, p=1.0),
])
norm = A.Compose([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

model.eval(); recalls = []
with torch.no_grad():
    for img_path, mask_path in val_pairs:
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        msk = (cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127).astype("uint8")
        out = occ_aug(image=img, mask=msk)
        occluded_img, msk_r = out["image"], out["mask"]
        occ_box = (occluded_img.sum(axis=2) == 0).astype("uint8")
        x = norm(image=occluded_img)["image"].unsqueeze(0).to(device)
        pred = (torch.sigmoid(model(x))[0, 0].cpu().numpy() >= CONFIG["threshold"])
        recalls.append(occlusion_recall(
            torch.from_numpy(pred.astype("float32")),
            torch.from_numpy(msk_r.astype("float32")),
            torch.from_numpy(occ_box.astype("float32"))))
print("Occlusion-Recall (val):", round(float(np.mean(recalls)), 4))

## 9 · Save off-device & visualise

Cloud sessions are wiped — copy the checkpoint to **Drive / a Kaggle Dataset**.

In [ ]:
# 9a · Persist the checkpoint off-device
ckpt = CKPT_DIR / "segformer_mitb0_deepglobe.pt"
print("checkpoint:", ckpt, "| exists:", ckpt.exists())
# Colab -> Drive:  from google.colab import drive; drive.mount('/content/drive')
#                  then copy {ckpt} into /content/drive/MyDrive/
# Kaggle -> it persists under /kaggle/working/ and can be saved as a Dataset.

In [ ]:
# 9b · Sanity-check predict_mask on a validation tile (the predict(tile) API)
import matplotlib.pyplot as plt
img_path, mask_path = val_pairs[0]
img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (CONFIG["tile_size"], CONFIG["tile_size"]))
pred = predict_mask(model, img, device=device, threshold=CONFIG["threshold"])
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img); ax[0].set_title("tile"); ax[0].axis("off")
ax[1].imshow(pred, cmap="gray"); ax[1].set_title("predicted mask"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print("predicted mask unique values:", np.unique(pred))

## Next (A5)

Load the saved checkpoint on CPU with `load_checkpoint(...)` and call
`predict_mask(model, tile)` per tile to produce `data/interim/{aoi}_mask.png`,
which P2 (Shaivi) consumes. That wiring is the walking-skeleton task **A5**.